# Train a neural network rocket controller

Train a small network to imitate the expert's continuous thrust and gimbal commands. Use this notebook for analysis and experiments: try different network sizes, learning rates, training data or losses, and record what you find. Low prediction error does not guarantee a safe landing: evaluate the saved controller in the simulator too.

Run this notebook from the practical-exercise folder. Install the local dependencies if needed: `python -m pip install torch numpy pandas matplotlib jupyter`. PyTorch is used for training; the browser runs the exported network with NumPy.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from scaffold import Controller, FEATURES, TARGETS, create_model

landings_path = "landings-varied.csv"
weights_path = "weights.npz"
seed = 42
learning_rate = 0.001
epochs = 30
batch_size = 256
np.random.seed(seed)
torch.manual_seed(seed)

In [ ]:
landings = pd.read_csv(landings_path)
print(f"Loaded {len(landings):,} observations from {landings.run_id.nunique()} flights")
display(landings.head())
display(landings[FEATURES + TARGETS].describe())

## Split by flight

Observations from the same flight are strongly related. Keep whole `run_id` groups together so validation and test flights are unseen during training. The `success` column describes the final outcome and is not available as an input during a flight. You can experiment with filtering failed demonstrations using training data only.

In [ ]:
run_ids = np.random.default_rng(seed).permutation(landings.run_id.unique())
train_end = int(0.8 * len(run_ids))
val_end = int(0.9 * len(run_ids))
train = landings[landings.run_id.isin(run_ids[:train_end])].copy()
val = landings[landings.run_id.isin(run_ids[train_end:val_end])].copy()
test = landings[landings.run_id.isin(run_ids[val_end:])].copy()
for name, frame in [("Train", train), ("Validation", val), ("Test", test)]:
    print(f"{name}: {frame.run_id.nunique()} flights, {len(frame):,} observations")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, target in zip(axes, TARGETS):
    ax.hist(train[target], bins=40)
    ax.set(title=f"Training {target}", xlabel=target, ylabel="Observations")
plt.tight_layout()
plt.show()

## Prepare and train

Standardize each input using only the training flights. The network predicts both commands with mean squared error; its sigmoid and tanh outputs keep predictions within the simulator's permitted ranges.

In [ ]:
mean = train[FEATURES].mean().to_numpy(dtype=np.float32)
std = train[FEATURES].std(ddof=0).to_numpy(dtype=np.float32)
std[std < 1e-8] = 1.0

def tensors(frame):
    x = (frame[FEATURES].to_numpy(dtype=np.float32) - mean) / std
    y = frame[TARGETS].to_numpy(dtype=np.float32)
    return torch.from_numpy(x), torch.from_numpy(y)

train_x, train_y = tensors(train)
val_x, val_y = tensors(val)
test_x, test_y = tensors(test)
loader = DataLoader(TensorDataset(train_x, train_y), batch_size=batch_size, shuffle=True)
model = create_model()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = nn.MSELoss()

In [ ]:
train_loss, val_loss = [], []
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for x_batch, y_batch in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(x_batch), y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(x_batch)
    train_loss.append(total_loss / len(train_x))
    model.eval()
    with torch.no_grad():
        val_loss.append(loss_fn(model(val_x), val_y).item())
    print(f"Epoch {epoch + 1:3d}: train {train_loss[-1]:.5f}, validation {val_loss[-1]:.5f}")

In [ ]:
plt.plot(train_loss, label="Training")
plt.plot(val_loss, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Mean squared error")
plt.legend()
plt.show()

## Evaluate and export

Compare errors for each output on the held-out flights. After choosing your model using the training and validation sets, use the test set for a final check. Upload both `scaffold.py` and `weights.npz` to the exercise page to evaluate the controller in closed-loop flights.

In [ ]:
model.eval()
with torch.no_grad():
    predictions = model(test_x).numpy()
errors = np.abs(predictions - test_y.numpy())
for i, target in enumerate(TARGETS):
    print(f"{target} test MAE: {errors[:, i].mean():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, (ax, target) in enumerate(zip(axes, TARGETS)):
    ax.scatter(test_y.numpy()[::10, i], predictions[::10, i], s=3, alpha=0.2)
    ax.set(xlabel="Expert", ylabel="Prediction", title=target)
plt.tight_layout()
plt.show()

In [ ]:
controller = Controller()
controller.save(weights_path, model, mean, std)
controller.load(weights_path)
print(f"Saved {weights_path}; example action: {controller.predict_action(test[FEATURES].iloc[0].to_numpy())}")